<a href="https://colab.research.google.com/github/rakeshkumarrs727-design/Python-Calculator/blob/main/DSP%20Source%20Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib, os, time

def h(*parts):
    m = hashlib.sha256()
    for p in parts:
        m.update(p if isinstance(p, bytes) else str(p).encode())
        m.update(b"|")
    return m.digest()

def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

DELTA_T = 5  # seconds allowed clock difference


class Server:
    def __init__(self):
        self.KS = os.urandom(32)
        self.users = {}   # PID -> dict(ID, revoked, locks)
        self.seen = set()  # nonce cache vs replay

    def register_lock(self, lock_id):
        return h(lock_id, self.KS)  # K_L

    def register_user(self, uid, hpw, locks):
        A = h(uid, self.KS)
        B = xor(A, hpw)
        V = h(uid, hpw, A)
        pid = h("PID", uid, os.urandom(8)).hex()[:16]
        self.users[pid] = dict(ID=uid, revoked=False, locks=locks)
        return dict(B=B, V=V, PID=pid)  # smart card content

    def handle(self, m2):
        lock_id, pid, N1, T1, P1, N2, T2, P2 = m2
        if abs(time.time() - T2) > DELTA_T or abs(time.time() - T1) > DELTA_T:
            return "REJECT: stale timestamp"
        if (N1, N2) in self.seen:
            return "REJECT: replayed nonce"
        KL = h(lock_id, self.KS)
        if h(lock_id, KL, P1, N2, T2) != P2:
            return "REJECT: lock authentication failed"
        u = self.users.get(pid)
        if not u:
            return "REJECT: unknown pseudonym"
        A = h(u["ID"], self.KS)
        if h(pid, A, N1, T1) != P1:
            return "REJECT: user authentication failed"
        if u["revoked"] or lock_id not in u["locks"]:
            return "REJECT: not authorised"
        self.seen.add((N1, N2))
        N3 = os.urandom(8)
        SK = h(A, N1, N2, N3, lock_id)
        enc_SK = xor(SK, h(KL, N2, N3))          # lightweight encryption of SK for the lock
        P3 = h(KL, N2, N3, SK, "OK")             # server -> lock
        P4 = h(A, N1, N2, N3, SK)                # server -> user
        return ("OK", N3, enc_SK, P3, P4)


class Lock:
    def __init__(self, lock_id, KL):
        self.id, self.KL, self.fails = lock_id, KL, 0

    def forward(self, m1):
        pid, N1, T1, P1 = m1
        N2, T2 = os.urandom(8), time.time()
        P2 = h(self.id, self.KL, P1, N2, T2)
        self.N2 = N2
        return (self.id, pid, N1, T1, P1, N2, T2, P2)

    def finish(self, resp):
        if isinstance(resp, str):
            self.fails += 1
            return False, resp, None
        _, N3, enc_SK, P3, P4 = resp
        SK = xor(enc_SK, h(self.KL, self.N2, N3))
        if h(self.KL, self.N2, N3, SK, "OK") != P3:
            return False, "REJECT: server tag invalid", None
        return True, "DOOR UNLOCKED", (self.N2, N3, P4, SK)


class UserCard:
    def __init__(self, uid, pw, bio, card, tau=None):
        self.uid, self.card = uid, card
        self.sigma = h("fuzzy", bio)  # simulated Rep(BIO', tau)
        self.pw = pw

    def login(self, pw=None, bio=None):
        sigma = self.sigma if bio is None else h("fuzzy", bio)
        hpw = h(self.uid, pw or self.pw, sigma)
        A = xor(self.card["B"], hpw)
        if h(self.uid, hpw, A) != self.card["V"]:
            return None  # local check fails: card never contacts network
        N1, T1 = os.urandom(8), time.time()
        P1 = h(self.card["PID"], A, N1, T1)
        self.A, self.N1 = A, N1
        return (self.card["PID"], N1, T1, P1)

    def verify_lock(self, N2, N3, P4, lock_id):
        SK = h(self.A, self.N1, N2, N3, lock_id)
        return h(self.A, self.N1, N2, N3, SK) == P4


def setup():
    S = Server()
    KL = S.register_lock("ROOM-101")
    L = Lock("ROOM-101", KL)
    uid, pw, bio = "alice", "S3cret!", "alice-fingerprint"
    hpw = h(uid, pw, h("fuzzy", bio))
    card = S.register_user(uid, hpw, ["ROOM-101"])
    return S, L, UserCard(uid, pw, bio, card), card


def run(S, L, U, **kw):
    m1 = U.login(**kw)
    if m1 is None:
        return False, "REJECT: local card check failed (wrong PW/biometric)"
    ok, msg, extra = L.finish(S.handle(L.forward(m1)))
    if ok:
        N2, N3, P4, _ = extra
        if not U.verify_lock(N2, N3, P4, L.id):
            return False, "User rejects lock (mutual auth failed)"
    return ok, msg


if __name__ == "__main__":
    results = []
    def t(name, outcome, expected_block=True):
        verdict = "PASS" if (outcome[0] is False) == expected_block else "FAIL"
        results.append((verdict, name, outcome[1]))

    S, L, U, card = setup()
    t("Legitimate user (PW + card + biometric)", run(S, L, U), expected_block=False)

    S, L, U, card = setup()
    t("Stolen card, wrong password", run(S, L, U, pw="guess123"))
    S, L, U, card = setup()
    t("Stolen card + password, wrong fingerprint", run(S, L, U, bio="attacker-finger"))

    # Replay: attacker records valid m1 and re-sends later
    S, L, U, card = setup()
    m1 = U.login()
    m2 = L.forward(m1)
    S.handle(m2)  # honest run consumed
    t("Replay of captured message M2", (False, S.handle(m2)))

    # Stale replay after delay
    S, L, U, card = setup()
    m1 = U.login()
    m2 = list(L.forward(m1)); m2[3] = time.time() - 60
    t("Old timestamp (delayed/replayed M1)", (False, S.handle(tuple(m2))))

    # Tampering (MITM modifies P1)
    S, L, U, card = setup()
    m1 = list(U.login()); m1[3] = h("forged")
    t("MITM modifies user tag P1", (False, S.handle(L.forward(tuple(m1)))))

    # Fake lock (no K_L)
    S, L, U, card = setup()
    fake = Lock("ROOM-101", os.urandom(32))
    t("Fake/cloned lock without K_L", (False, S.handle(fake.forward(U.login()))))

    # Impersonation with random pseudonym
    S, L, U, card = setup()
    fm = (os.urandom(8), time.time(), h("x"))
    t("Forged user (guessed pseudonym)", (False, S.handle(L.forward(("deadbeef", *fm)))))

    # Revoked card
    S, L, U, card = setup()
    S.users[card["PID"]]["revoked"] = True
    t("Lost card after revocation by admin", run(S, L, U))

    # Unauthorised room
    S, L, U, card = setup()
    L2 = Lock("ROOM-202", S.register_lock("ROOM-202"))
    t("Valid user, room not in access list", run(S, L2, U))

    # Fake server response to lock
    S, L, U, card = setup()
    m1 = U.login(); L.forward(m1)
    bogus = ("OK", os.urandom(8), os.urandom(32), os.urandom(32), os.urandom(32))
    t("Fake server reply to lock", L.finish(bogus))

    for r in results:
        print(f"[{r[0]}] {r[1]:48s} -> {r[2]}")
    print("\nAll tests passed" if all(r[0] == "PASS" for r in results) else "\nSOME TESTS FAILED")


[PASS] Legitimate user (PW + card + biometric)          -> DOOR UNLOCKED
[PASS] Stolen card, wrong password                      -> REJECT: local card check failed (wrong PW/biometric)
[PASS] Stolen card + password, wrong fingerprint        -> REJECT: local card check failed (wrong PW/biometric)
[PASS] Replay of captured message M2                    -> REJECT: replayed nonce
[PASS] Old timestamp (delayed/replayed M1)              -> REJECT: stale timestamp
[PASS] MITM modifies user tag P1                        -> REJECT: user authentication failed
[PASS] Fake/cloned lock without K_L                     -> REJECT: lock authentication failed
[PASS] Forged user (guessed pseudonym)                  -> REJECT: unknown pseudonym
[PASS] Lost card after revocation by admin              -> REJECT: not authorised
[PASS] Valid user, room not in access list              -> REJECT: not authorised
[PASS] Fake server reply to lock                        -> REJECT: server tag invalid

All tests pass